In [ ]:
%cd ../.
import os, sys
sys.path.insert(0, os.path.abspath('../Scripts'))
sys.path.insert(0, os.path.expanduser('~/CDD_Vault_API/python'))  # CDD Vault API (get_df)

In [ ]:
%load_ext autoreload
%autoreload 2

import re
import os
import pandas as pd
import numpy as np
import py3Dmol
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import csv
import pickle

from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem,rdFMCS
# import prolif as plf
from glob import glob
import meeko
import subprocess as sub
# from vina import Vina
import time
from tqdm import tqdm
tqdm.pandas()
import importlib
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import StratifiedKFold, cross_val_predict, KFold
from sklearn.metrics import f1_score, silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef
from xgboost import XGBClassifier,XGBRegressor
from openTSNE import TSNE as oTSNE        # pip install openTSNE
import seaborn as sns
from scipy import stats
import csv, contextlib, threading, joblib
import joblib
from joblib import Parallel, delayed
from datetime import date
## load internal (in-house) + public-augmented data, champion model, eval helpers
from copy import deepcopy



# user defined modules
import Rdkit_tools as rdkit_tools
importlib.reload(rdkit_tools)
import Molecule as M
import ML_Reg as ML_Reg
import ML_Class as ML_Class
from MolViz3D import MolViz3D
import Statistics_tools as stats_tools
import python.functions as fn
from python.harmonize_sol import harmonize_solubility

from get_library import get_df   # CDD Vault collection export
from get_protocol_data import get_data, load_config, alias_map
# from tdc.multi_pred import DTI

In [ ]:
## params — single source of truth in config/config.yaml.
## Loaded as a `config` namespace AND injected as globals, so both
import yaml
from types import SimpleNamespace
with open('config/config.yaml') as _f:
    _cfg = yaml.safe_load(_f)
config = SimpleNamespace(**_cfg)
globals().update(_cfg)
print(f'> loaded {len(_cfg)} params from config/config.yaml')


## 0. Imports

In [ ]:
## paths are relative to this notebook (vignettes/)
CONFIG = '../CDD_Vault_API/config/config.yaml'
TOKEN_FILE = '../.cdd_token'   # a READ token is enough to extract

alias_map(load_config(CONFIG))

In [ ]:
## get all protocol data
df_all = get_data(config_path=CONFIG, token_file=TOKEN_FILE)
print('shape:', df_all.shape)
## save a local copy so that Claude can work on it
# df_all.to_csv('data/20260707_all_adme.csv',index=False)

In [ ]:
## ---- harmonized modelling data (pre-built caches, autoresearch/predict_adme Phase 1) ----
## Features (MF fingerprints + Descriptastorus) already computed in modelling space, so load the
## parquet caches directly — identical data to the autoresearch campaign (no re-featurizing needed).
CACHE = Path('autoresearch/predict_adme')
SEED, N_JOBS = 42, 32
CHAMPION_RF = dict(n_estimators=200, max_depth=20, max_features=0.3,
                   min_samples_leaf=2, n_jobs=N_JOBS, random_state=SEED)
ENDPOINTS = ['solubility', 'logd', 'hlm', 'mlm', 'rlm', 'caco2', 'mdck', 'ppb']
PUBLIC_SRC = {'EXP': 'public_{ep}.parquet', 'NVS': 'public_novartis_{ep}.parquet', 'ADM': 'public_admetlab_{ep}.parquet'}

_tgt = pd.read_parquet(CACHE / 'internal_targets.parquet')                       # in-house: compound, smiles, 8 endpoints
_mf  = pd.read_parquet(CACHE / 'internal_MF.parquet').drop_duplicates('compound')
_ds  = pd.read_parquet(CACHE / 'internal_DS.parquet').drop_duplicates('compound')
FEATS = [c for c in _mf.columns if c != 'compound'] + [c for c in _ds.columns if c != 'compound']  # MF + DS
internal = _tgt.merge(_mf, on='compound').merge(_ds, on='compound')

## global-temporal test = newest 30% of ALL in-house compounds by SRB id (same test compounds across models)
_srb = _tgt['compound'].str.extract(r'(\d+)')[0].astype(float).to_numpy()
GLOBAL_TEST = set(_tgt['compound'].to_numpy()[np.argsort(_srb)[int(len(_tgt) * 0.7):]])
print(f'internal {internal.shape[0]} cmpd | {len(FEATS)} MF+DS features | global-temporal test = {len(GLOBAL_TEST)} newest')


def pooled_ML_data(endpoint, sources=()):
    """Build one ML_data (internal + chosen public sources) + global-temporal ID_sets for
    ML_Reg.K_fold_by_defined_IDs. label = endpoint value; public rows all go to TRAIN;
    test = internal newest-30% compounds with the endpoint measured. sources ⊆ {EXP, NVS, ADM}."""
    d = (internal[['compound', 'smiles', endpoint]].dropna(subset=[endpoint]).rename(columns={endpoint: 'label'})
                 .merge(internal[['compound'] + FEATS], on='compound'))
    parts = [d]
    for s in sources:
        pub = pd.read_parquet(CACHE / PUBLIC_SRC[s].format(ep=endpoint))
        parts.append(pub[['compound', 'smiles', 'value'] + FEATS].rename(columns={'value': 'label'}))
    ML_data = pd.concat(parts, ignore_index=True)
    in_test = ML_data['compound'].isin(GLOBAL_TEST)
    ID_sets = [[ML_data.loc[~in_test, 'compound'].tolist(),
                d.loc[d['compound'].isin(GLOBAL_TEST), 'compound'].tolist()]]
    return ML_data, ID_sets

## 1. Evaluate models

Reproduce the **deployment verdict** — for each endpoint, train its best model + dataset combination and report performance on the identical **global-temporal** test (in-house Pearson r²), via `ML_Reg.K_fold_by_defined_IDs`.

| endpoint | best config | target R² |
|---|---|---|
| logd | RF single-task (MF+DS, +EXP+ADM) | 0.42 |
| mdck | RF single-task | 0.32 |
| solubility | Chemprop `{sol,logd}` group | 0.30 |
| mlm | Chemprop all-8 | 0.385 |
| hlm | Chemprop all-8 + Chemeleon | 0.20 |
| ppb | weak everywhere | ~0.05 |

RF models are trained live below; Chemprop models are retrained live (GPU) when we reach them.

### 1.1 logd — RF single-task (MF+DS)

Deployment pick for **logd** (verdict table): champion RandomForest on MF+DS features, in-house data pooled with **experimental** (TDC `Lipophilicity_AstraZeneca`) + **ADMETlab** public. Scored on the matched global-temporal test (newest 30% of in-house compounds). Target ≈ **R² 0.42**.

In [ ]:
## logd — RF single-task on MF+DS. Two arms (internal-only vs + public EXP+ADM), two evals:
##   temporal      = predict newest-30% held-out compounds (prospective/extrapolation)
##   5-fold CV     = random interpolation; folds defined on INTERNAL compounds only (identical
##                   folds both arms), public rows always in TRAIN (never in a test fold).
def _rf_temporal(endpoint, sources):
    ML_data, ID_sets = pooled_ML_data(endpoint, sources=sources)
    _rf, pred = ML_Reg.K_fold_by_defined_IDs(
        ML_data, ID='compound', ID_sets=ID_sets,
        model=RandomForestRegressor(**CHAMPION_RF), col_to_rm=['compound', 'smiles', 'label'], v=False)
    return pred, stats_tools.rsquared(pred['real_y'], pred['pred_y']), stats.spearmanr(pred['real_y'], pred['pred_y']).statistic

def _rf_cv(endpoint, sources, folds=5):
    d = (internal[['compound', 'smiles', endpoint]].dropna(subset=[endpoint]).rename(columns={endpoint: 'label'})
                 .merge(internal[['compound'] + FEATS], on='compound'))
    pub = [pd.read_parquet(CACHE / PUBLIC_SRC[s].format(ep=endpoint))[['compound', 'smiles', 'value'] + FEATS]
             .rename(columns={'value': 'label'}) for s in sources]
    ML_data = pd.concat([d] + pub, ignore_index=True)
    pub_ids = pd.concat(pub)['compound'].tolist() if pub else []
    ids = d['compound'].to_numpy()
    kf = KFold(n_splits=folds, shuffle=True, random_state=SEED)
    ID_sets = [[list(ids[tr]) + pub_ids, list(ids[te])] for tr, te in kf.split(ids)]   # test = internal only
    _rf, pred = ML_Reg.K_fold_by_defined_IDs(
        ML_data, ID='compound', ID_sets=ID_sets,
        model=RandomForestRegressor(**CHAMPION_RF), col_to_rm=['compound', 'smiles', 'label'], v=False)
    return pred, stats_tools.rsquared(pred['real_y'], pred['pred_y']), stats.spearmanr(pred['real_y'], pred['pred_y']).statistic

pred_int,    r2_int,    rho_int    = _rf_temporal('logd', sources=())
pred,        r2,        rho        = _rf_temporal('logd', sources=('EXP', 'ADM'))
pred_cv_int, r2_cv_int, rho_cv_int = _rf_cv('logd', sources=())
pred_cv,     r2_cv,     rho_cv     = _rf_cv('logd', sources=('EXP', 'ADM'))
print(f'logd  internal-only     temporal  R2={r2_int:.3f} rho={rho_int:.3f} n={len(pred_int)}   |  5-fold CV  R2={r2_cv_int:.3f} rho={rho_cv_int:.3f} n={len(pred_cv_int)}')
print(f'logd  + public EXP+ADM  temporal  R2={r2:.3f} rho={rho:.3f} n={len(pred)}   |  5-fold CV  R2={r2_cv:.3f} rho={rho_cv:.3f} n={len(pred_cv)}')

In [ ]:
## side-by-side parity: internal-only (ember) vs + public EXP+ADM (azure), same global-temporal test
fig, axes = plt.subplots(1, 2, figsize=(11, 5.5), sharex=True, sharey=True)
for ax, (p, r2_, rho_, name, color) in zip(axes, [
        (pred_int, r2_int, rho_int, 'internal only',      SERAC_C['ember']),
        (pred,     r2,     rho,     '+ public (EXP+ADM)', SERAC_C['azure'])]):
    ax.scatter(p['real_y'], p['pred_y'], c=color, s=6, alpha=0.7)
    ax.plot([0, 6], [0, 6], ls='--', lw=0.8, c='#888')
    ax.set_xlim([0, 6]); ax.set_ylim([0, 6]); ax.set_aspect('equal')
    ax.set_xlabel('ground truth'); ax.set_ylabel('prediction')
    ax.set_title(f'logd — {name}\nR²={r2_:.3f}  ρ={rho_:.3f}  n={len(p)}')
fig.tight_layout()

In [16]:
## solubility — RF single-task on MF+DS, scored on the SAME global-temporal test as Chemprop.
## Arms: internal-only vs + EXP (experimental, thermo+kinetic). NB: ADMETlab (ADM) is PREDICTED
## PROTAC-patent data that POISONS solubility (~0.31 -> ~0.02) — dropped here; uncomment to see it.
## (_rf_temporal / _rf_cv are defined in the logd cell above.)
pred_int,    r2_int,    rho_int    = _rf_temporal('solubility', sources=())
pred,        r2,        rho        = _rf_temporal('solubility', sources=('EXP',))
# pred, r2, rho = _rf_temporal('solubility', sources=('EXP', 'ADM'))   # ADMETlab poisons solubility -> ~0.02
pred_cv_int, r2_cv_int, rho_cv_int = _rf_cv('solubility', sources=())
pred_cv,     r2_cv,     rho_cv     = _rf_cv('solubility', sources=('EXP',))
print(f'solubility  internal-only  temporal  R2={r2_int:.3f} rho={rho_int:.3f} n={len(pred_int)}   |  5-fold CV  R2={r2_cv_int:.3f} rho={rho_cv_int:.3f} n={len(pred_cv_int)}')
print(f'solubility  + public EXP   temporal  R2={r2:.3f} rho={rho:.3f} n={len(pred)}   |  5-fold CV  R2={r2_cv:.3f} rho={rho_cv:.3f} n={len(pred_cv)}')
print('reference: Chemprop {sol,logd} group = 0.30 on this same global-temporal test')

100%|██████████| 5/5 [15:13<00:00, 182.64s/it]

solubility  internal-only  temporal  R2=0.180 rho=0.373 n=91   |  5-fold CV  R2=0.689 rho=0.779 n=120
solubility  + public EXP   temporal  R2=0.254 rho=0.509 n=91   |  5-fold CV  R2=0.736 rho=0.803 n=120
reference: Chemprop {sol,logd} group = 0.30 on this same global-temporal test
